##### config

In [1]:
%run 00_config

StatementMeta(, 236ff7eb-4aa0-47bd-8e07-8989e3d6ca21, 3, Finished, Available, Finished, True)

FHIR Pipeline Config loaded
  FHIR URL   : https://hapi.fhir.org/baseR4
  Resources  : ['Patient', 'Encounter', 'Observation', 'Condition']
  Ingest days: 3
  Schemas    : bronze | silver | gold
  Naming     : tables → tbl_  | views → vw_

Table names:
  bronze.tbl_patient                  → silver.tbl_patient                  → gold.vw_dim_patient
  bronze.tbl_encounter                → silver.tbl_encounter                → gold.vw_fact_encounter
  bronze.tbl_observation              → silver.tbl_observation              → gold.vw_fact_observation
  bronze.tbl_condition                → silver.tbl_condition                → gold.vw_fact_condition


##### Imports and function

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import datetime

CURRENT_TS = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S")

def upsert_silver_scd2(resource: str, parsed_df):
    """
    SCD Type 2 upsert into silver.tbl_<resource>
    - New records     → inserted as current
    - Changed records → old row closed, new row inserted
    - Unchanged       → skipped
    """
    table_name   = silver_table(resource)
    business_key = CONFIG["business_key"]

    incoming = parsed_df \
        .withColumn("effective_from", F.lit(CURRENT_TS)) \
        .withColumn("effective_to",   F.lit(CONFIG["high_date"])) \
        .withColumn("is_current",     F.lit(True)) \
        .withColumn("row_hash", F.md5(
            F.concat_ws("|", *[c for c in parsed_df.columns if c != business_key])
        ))

    try:
        spark.table(table_name)
        table_exists = True
    except:
        table_exists = False

    if table_exists:
        silver_dt = DeltaTable.forName(spark, table_name)

        # Step 1 — close out changed rows
        silver_dt.alias("old").merge(
            incoming.alias("new"),
            f"old.{business_key} = new.{business_key} \
              AND old.is_current = true \
              AND old.row_hash != new.row_hash"
        ).whenMatchedUpdate(set={
            "is_current":   F.lit(False),
            "effective_to": F.lit(CURRENT_TS)
        }).execute()

        # Step 2 — insert new and changed rows only
        existing_current = spark.table(table_name) \
            .filter("is_current = true") \
            .select(business_key, "row_hash")

        new_rows = incoming.join(
            existing_current,
            on=[business_key, "row_hash"],
            how="left_anti"
        )
        new_rows.write.format("delta") \
            .mode("append") \
            .saveAsTable(table_name)
    else:
        # First run — create table fresh
        incoming.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(table_name)

    current = spark.table(table_name).filter("is_current = true").count()
    total   = spark.table(table_name).count()
    print(f"  {table_name}: {current} current | {total} total")

print(f"SCD2 function ready | run timestamp: {CURRENT_TS}")

StatementMeta(, 236ff7eb-4aa0-47bd-8e07-8989e3d6ca21, 4, Finished, Available, Finished, False)

SCD2 function ready | run timestamp: 2026-06-16T17:15:32


##### Silver Patient

In [3]:
print(f"=== {silver_table('Patient')} ===")

parsed = spark.table(bronze_table("Patient")) \
    .withColumn("p", F.from_json(F.col("raw_json"), MapType(StringType(), StringType()))) \
    .select(
        F.col("resource_id"),
        F.col("p.gender").alias("gender"),
        F.col("p.birthDate").alias("birth_date"),
        F.col("p.deceasedBoolean").alias("deceased"),
        F.col("extraction_timestamp"),
        F.col("load_date")
    ) \
    .dropDuplicates(["resource_id"])

upsert_silver_scd2("Patient", parsed)

StatementMeta(, 236ff7eb-4aa0-47bd-8e07-8989e3d6ca21, 5, Finished, Available, Finished, False)

=== silver.tbl_patient ===
  silver.tbl_patient: 634 current | 634 total


##### Silver Encounter

In [4]:
print(f"=== {silver_table('Encounter')} ===")

parsed = spark.table(bronze_table("Encounter")) \
    .withColumn("e", F.from_json(F.col("raw_json"), MapType(StringType(), StringType()))) \
    .select(
        F.col("resource_id"),
        F.col("e.status").alias("status"),
        F.col("e.class").alias("encounter_class"),
        F.col("extraction_timestamp"),
        F.col("load_date")
    ) \
    .dropDuplicates(["resource_id"])

upsert_silver_scd2("Encounter", parsed)

StatementMeta(, 236ff7eb-4aa0-47bd-8e07-8989e3d6ca21, 6, Finished, Available, Finished, False)

=== silver.tbl_encounter ===
  silver.tbl_encounter: 273 current | 273 total


##### Silver Observation

In [5]:
print(f"=== {silver_table('Observation')} ===")

parsed = spark.table(bronze_table("Observation")) \
    .withColumn("o", F.from_json(F.col("raw_json"), MapType(StringType(), StringType()))) \
    .select(
        F.col("resource_id"),
        F.col("o.status").alias("status"),
        F.col("o.effectiveDateTime").alias("effective_date"),
        F.col("o.subject").alias("subject_ref"),
        F.col("extraction_timestamp"),
        F.col("load_date")
    ) \
    .dropDuplicates(["resource_id"])

upsert_silver_scd2("Observation", parsed)

StatementMeta(, 236ff7eb-4aa0-47bd-8e07-8989e3d6ca21, 7, Finished, Available, Finished, False)

=== silver.tbl_observation ===
  silver.tbl_observation: 750 current | 750 total


##### Silver Condition

In [6]:
print(f"=== {silver_table('Condition')} ===")

parsed = spark.table(bronze_table("Condition")) \
    .withColumn("c", F.from_json(F.col("raw_json"), MapType(StringType(), StringType()))) \
    .select(
        F.col("resource_id"),
        F.col("c.clinicalStatus").alias("clinical_status"),
        F.col("c.verificationStatus").alias("verification_status"),
        F.col("c.recordedDate").alias("recorded_date"),
        F.col("c.subject").alias("subject_ref"),
        F.col("extraction_timestamp"),
        F.col("load_date")
    ) \
    .dropDuplicates(["resource_id"])

upsert_silver_scd2("Condition", parsed)

StatementMeta(, 236ff7eb-4aa0-47bd-8e07-8989e3d6ca21, 8, Finished, Available, Finished, False)

=== silver.tbl_condition ===
  silver.tbl_condition: 679 current | 679 total
